# 05 - Evaluation of all models
Reads every `results/**/preds_*.parquet`, picks each model's decision threshold **on validation**, and reports everything.

**Order of work:** run Section A (validation only) until you are happy. When all models are final, set `RUN_TEST = True` and run Section B **once**. Do not change models after looking at test numbers.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from paths import *
print("repo root:", ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

FINAL.mkdir(parents=True, exist_ok=True)
meta = {s: pd.read_parquet(PROCESSED / f"{s}.parquet") for s in ["train", "val", "test"]}
ID_COL = detect_id_col(meta["train"])
CODE = code_col(meta["train"])
LLM_COL = next((c for c in ["llm_name", "llm"] if c in meta["train"].columns), None)
SCEN_COL = "scenario" if "scenario" in meta["train"].columns else None
PROB_COL = "problem_id" if "problem_id" in meta["train"].columns else None
print("id:", ID_COL, "| code:", CODE, "| llm:", LLM_COL, "| scenario:", SCEN_COL, "| problem:", PROB_COL)

## Load prediction files and add the two reference models

In [ ]:
preds = {}
for f in sorted(RESULTS.rglob("preds_*.parquet")):
    p = pd.read_parquet(f)
    name = f.stem.replace("preds_", "", 1)
    if ID_COL in p.columns and "sample_id" not in p.columns:
        p = p.rename(columns={ID_COL: "sample_id"})
    need = {"sample_id", "split", "y_true", "y_score"}
    if need - set(p.columns):
        print("SKIPPED", name, "- missing columns:", sorted(need - set(p.columns)), "| has:", p.columns.tolist())
        continue
    p["split"] = p["split"].replace({"validation": "val", "valid": "val"})
    p["sample_id"] = p["sample_id"].astype(str)
    preds[name] = p[["sample_id", "split", "y_true", "y_score"]].copy()
print("models found:", list(preds))

# reference 1: always predict human.  reference 2: length only (train on train).
def length_X(df):
    s = df[CODE].astype(str)
    return np.c_[np.log1p(s.str.len()), np.log1p(s.str.count("\n") + 1)]

lr_len = LogisticRegression(class_weight="balanced", random_state=SEED)
lr_len.fit(length_X(meta["train"]), to_binary_label(meta["train"][LABEL]))
refs = {"always_human": lambda df: np.zeros(len(df)),
        "length_only": lambda df: lr_len.predict_proba(length_X(df))[:, 1]}
for name, fn in refs.items():
    frames = []
    for s in ["val", "test"]:
        df = meta[s]
        frames.append(pd.DataFrame({"sample_id": df[ID_COL].astype(str).to_numpy(), "split": s,
                                    "y_true": to_binary_label(df[LABEL]), "y_score": fn(df)}))
    preds[name] = pd.concat(frames, ignore_index=True)
REAL = [m for m in preds if m not in refs]
print("real models:", REAL)

## Section A - validation only

In [ ]:
def pick_threshold(y, s):
    """Threshold that maximises F1 on validation."""
    s = np.asarray(s, dtype=float)
    if np.unique(s).size == 1:
        return float(s[0]) + 1.0          # constant scores: predict nobody as AI
    grid = np.unique(np.quantile(s, np.linspace(0.02, 0.98, 97)))
    f1s = [f1_score(y, (s >= t).astype(int), zero_division=0) for t in grid]
    return float(grid[int(np.argmax(f1s))])

def compute(y, s, thr):
    y = np.asarray(y); s = np.asarray(s, dtype=float)
    pred = (s >= thr).astype(int)
    return {"n": len(y),
            "acc": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "auc": roc_auc_score(y, s) if len(np.unique(y)) > 1 else np.nan}

thr, val_m = {}, {}
for name, p in preds.items():
    v = p[p.split == "val"]
    thr[name] = pick_threshold(v.y_true, v.y_score)
    val_m[name] = compute(v.y_true, v.y_score, thr[name])
val_table = pd.DataFrame(val_m).T.round(4)
val_table.insert(0, "threshold", pd.Series(thr).round(4))
display(val_table)

best = max(REAL, key=lambda m: (val_m[m]["f1"], val_m[m]["auc"]))
print("Best real model on validation (F1, then AUC):", best)

## Section B - final test run (do this once)

In [ ]:
RUN_TEST = False   # <- set to True only when every model is final
assert RUN_TEST, "Test set is locked. Finish Section A first, then set RUN_TEST = True and run from here once."

In [ ]:
rows = []
test_frames = {}
for name, p in preds.items():
    t = p[p.split == "test"].copy()
    t["y_pred"] = (t.y_score.to_numpy() >= thr[name]).astype(int)
    test_frames[name] = t
    m = compute(t.y_true, t.y_score, thr[name])
    rows.append({"model": name, "n_val": val_m[name]["n"], "val_acc": val_m[name]["acc"],
                 "val_f1": val_m[name]["f1"], "val_auc": val_m[name]["auc"],
                 "n_test": m["n"], "test_acc": m["acc"], "test_precision": m["precision"],
                 "test_recall": m["recall"], "test_f1": m["f1"], "test_auc": m["auc"],
                 "threshold": thr[name]})
comparison = pd.DataFrame(rows).set_index("model").round(4)
comparison.to_csv(FINAL / "comparison_table.csv")
display(comparison)

In [ ]:
# ROC curves (test) and confusion matrix of the best model
plt.figure(figsize=(6, 5))
for name, t in test_frames.items():
    if name == "always_human":
        continue
    fpr, tpr, _ = roc_curve(t.y_true, t.y_score)
    plt.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(t.y_true, t.y_score):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC curves - test set")
plt.legend(fontsize=7); plt.tight_layout()
plt.savefig(FINAL / "roc_curves.png", dpi=150); plt.show()

tb = test_frames[best]
cm = confusion_matrix(tb.y_true, tb.y_pred)
ConfusionMatrixDisplay(cm, display_labels=["human", "AI"]).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion matrix - {best} (test)"); plt.tight_layout()
plt.savefig(FINAL / "confusion_matrix.png", dpi=150); plt.show()

In [ ]:
# Attach metadata (LLM, scenario, length) to the test predictions
extra_cols = [c for c in [LLM_COL, SCEN_COL, PROB_COL] if c]
mt = meta["test"].copy()
mt["sample_id"] = mt[ID_COL].astype(str)
mt["n_chars_eval"] = mt[CODE].astype(str).str.len()
mt = mt[["sample_id", "n_chars_eval", CODE] + extra_cols]

def attach(name):
    return test_frames[name].merge(mt, on="sample_id", how="left")

def recall_by(col):
    out = {}
    for name in test_frames:
        if name == "always_human":
            continue
        d = attach(name)
        ai = d[d.y_true == 1]
        out[name] = ai.assign(hit=(ai.y_pred == 1)).groupby(col)["hit"].mean()
    return pd.DataFrame(out).T.round(3)

if LLM_COL:
    per_llm = recall_by(LLM_COL)
    per_llm.to_csv(FINAL / "breakdown_per_llm.csv")
    print("AI recall per LLM (test):"); display(per_llm)
    print("AI samples per LLM:", attach(best).query("y_true == 1")[LLM_COL].value_counts().to_dict())
if SCEN_COL:
    per_scen = recall_by(SCEN_COL)
    per_scen.to_csv(FINAL / "breakdown_per_scenario.csv")
    print("AI recall per scenario (test):"); display(per_scen)

In [ ]:
# Accuracy by length bin (does length drive the result?)
d = attach(best)
d["length_bin"] = pd.qcut(d["n_chars_eval"], 5, duplicates="drop")
by_len = d.groupby("length_bin", observed=True).apply(
    lambda g: pd.Series({"n": len(g), "accuracy": (g.y_true == g.y_pred).mean(),
                         "ai_recall": ((g.y_pred == 1) & (g.y_true == 1)).sum() / max((g.y_true == 1).sum(), 1)}),
    include_groups=False).round(3)
by_len.to_csv(FINAL / "accuracy_by_length.csv")
print(f"Best model: {best}"); display(by_len)

In [ ]:
# Leakage checks
lines = []
if PROB_COL:
    ids = {s: set(meta[s][PROB_COL].astype(str)) for s in meta}
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        lines.append(f"shared problem_ids {a}/{b}: {len(ids[a] & ids[b])}")
if "code_hash" in meta["train"].columns:
    hs = {s: set(meta[s]["code_hash"].astype(str)) for s in meta}
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        lines.append(f"shared code_hash {a}/{b}: {len(hs[a] & hs[b])}")
else:
    lines.append("code_hash column not in split files: hash check skipped")
(FINAL / "leakage_check.txt").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

In [ ]:
# Error look: the most confident mistakes of the best model (2 AI called human, 3 human called AI)
d = attach(best)
ai_missed = d[(d.y_true == 1) & (d.y_pred == 0)].sort_values("y_score").head(2)
hu_flagged = d[(d.y_true == 0) & (d.y_pred == 1)].sort_values("y_score", ascending=False).head(3)
err = pd.concat([ai_missed.assign(error="AI called human"), hu_flagged.assign(error="human called AI")])
err["code_preview"] = err[CODE].astype(str).str.slice(0, 600)
err["why_guess"] = ""   # fill this in by hand: one line per example
keep = ["error", "sample_id", "y_score"] + [c for c in [LLM_COL, SCEN_COL] if c] + ["code_preview", "why_guess"]
err[keep].to_csv(FINAL / "error_examples.csv", index=False)
for _, r in err.iterrows():
    print("=" * 70); print(r["error"], "| score", round(r["y_score"], 3), "|", r.get(LLM_COL, ""), r.get(SCEN_COL, ""))
    print(r["code_preview"])

In [ ]:
summary = {
    "best_model_on_validation": best,
    "thresholds_chosen_on_validation": {k: round(v, 5) for k, v in thr.items()},
    "test_metrics": comparison[["n_test", "test_acc", "test_precision", "test_recall", "test_f1", "test_auc"]].to_dict("index"),
    "validation_metrics": comparison[["n_val", "val_acc", "val_f1", "val_auc"]].to_dict("index"),
    "leakage_check": lines,
}
(FINAL / "final_results.json").write_text(json.dumps(summary, indent=2, default=float), encoding="utf-8")
print("saved results/final/final_results.json")
print([p.name for p in sorted(FINAL.iterdir())])